In [0]:
HADNLING THE DATA

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, split, to_date, hour,
    current_timestamp
)
from pyspark.sql.types import IntegerType, DoubleType

# --------------------------------------------------
# Step 1: Create Spark Session
# --------------------------------------------------
spark = SparkSession.builder \
    .appName("HealthcareDataProcessing") \
    .getOrCreate()

# --------------------------------------------------
# Step 2: Load Data
# --------------------------------------------------
vitals_df = spark.read.csv(
    "patient_vitals.csv",
    header=True,
    inferSchema=True
)

details_df = spark.read.csv(
    "patient_details.csv",
    header=True,
    inferSchema=True
)

device_df = spark.read.csv(
    "device_logs.csv",
    header=True,
    inferSchema=True
)

# --------------------------------------------------
# Step 3: Data Cleaning
# --------------------------------------------------

# Remove duplicate records
vitals_df = vitals_df.dropDuplicates()

# Handle null readings
vitals_df = vitals_df.fillna({
    "heart_rate": 0,
    "oxygen_level": 0,
    "temperature": 0
})

# Remove records with missing patient_id
vitals_df = vitals_df.filter(col("patient_id").isNotNull())

# Handle missing patient information
details_df = details_df.fillna({
    "name": "Unknown",
    "disease": "Unknown",
    "city": "Unknown",
    "gender": "Unknown"
})

# --------------------------------------------------
# Step 4: Correct Incorrect Values
# --------------------------------------------------

# Heart Rate Range: 30 - 220
vitals_df = vitals_df.withColumn(
    "heart_rate",
    when(
        (col("heart_rate") < 30) | (col("heart_rate") > 220),
        None
    ).otherwise(col("heart_rate"))
)

# Oxygen Level Range: 0 - 100
vitals_df = vitals_df.withColumn(
    "oxygen_level",
    when(
        (col("oxygen_level") < 0) | (col("oxygen_level") > 100),
        None
    ).otherwise(col("oxygen_level"))
)

# Temperature Range: 90 - 110 F
vitals_df = vitals_df.withColumn(
    "temperature",
    when(
        (col("temperature") < 90) | (col("temperature") > 110),
        None
    ).otherwise(col("temperature"))
)

# Blood Pressure Validation
bp_parts = split(col("blood_pressure"), "/")

vitals_df = vitals_df.withColumn(
    "systolic",
    bp_parts.getItem(0).cast(IntegerType())
).withColumn(
    "diastolic",
    bp_parts.getItem(1).cast(IntegerType())
)

vitals_df = vitals_df.filter(
    (col("systolic").between(70, 250)) &
    (col("diastolic").between(40, 150))
)

# --------------------------------------------------
# Step 5: Data Transformation
# --------------------------------------------------

vitals_df = vitals_df.withColumn(
    "health_status",
    when(col("oxygen_level") < 90, "Critical")
    .when(col("temperature") > 100, "Fever")
    .when(col("heart_rate") > 100, "High Risk")
    .otherwise("Normal")
)

vitals_df = vitals_df.withColumn(
    "alert_flag",
    when(col("health_status") != "Normal", "Y")
    .otherwise("N")
)

vitals_df = vitals_df.withColumn(
    "monitoring_date",
    to_date(col("timestamp"))
)

vitals_df = vitals_df.withColumn(
    "monitoring_hour",
    hour(col("timestamp"))
)

# --------------------------------------------------
# Step 6: Join Datasets
# --------------------------------------------------

health_df = vitals_df.alias("v") \
    .join(
        details_df.alias("p"),
        col("v.patient_id") == col("p.patient_id"),
        "left"
    ) \
    .join(
        device_df.alias("d"),
        col("v.patient_id") == col("d.patient_id"),
        "left"
    )

# --------------------------------------------------
# Step 7: Select Final Output Columns
# --------------------------------------------------

final_df = health_df.select(
    col("v.patient_id").alias("patient_id"),
    col("p.name").alias("patient_name"),
    col("p.disease").alias("disease"),
    col("v.heart_rate"),
    col("v.oxygen_level"),
    col("v.temperature"),
    col("v.health_status"),
    col("d.device_type"),
    col("v.monitoring_date")
)

# --------------------------------------------------
# Step 8: Show Final Data
# --------------------------------------------------

final_df.show(truncate=False)

# --------------------------------------------------
# Step 9: Store Processed Data
# --------------------------------------------------

# Save as Parquet
final_df.write \
    .mode("overwrite") \
    .parquet("output/processed_healthcare_data")

# Optional: Save as CSV
final_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/processed_healthcare_csv")

print("Healthcare ETL Pipeline Completed Successfully")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:726)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:444)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:444)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can